## Spatial VQA generated from COCO dataset

`spatial_vqa.jsonl` contains automatically generated spatial visual question–answer pairs derived from COCO images and annotations.  
Each entry includes a spatial reasoning question, its answer, and structured supporting facts (e.g., object relations, distances) suitable for analysis and fine-tuning of VLMs/MLLMs.


### 1. Setup & Configuration

This section initializes the environment and defines global configuration parameters for dataset generation.

**Imports**
- `pycocotools`
- `PIL`
- `numpy`
- `json`
- `tqdm`
- `pandas`

**Paths**
- `COCO_ROOT`: root directory of the COCO dataset  
- `annFile`: path to COCO instance annotations (`instances_*.json`)  
- Output directory: `data/generated/coco_spatial_vqa/`

**Global Configuration**
- `N_IMAGES`: number of images to sample  
- `QUESTIONS_PER_IMAGE`: number of spatial questions generated per image  
- `MAX_OBJECTS_PER_IMAGE`: upper bound on detected objects per image  
- `MIN_BBOX_AREA`: minimum bounding-box area threshold  
- `MODEL_NAME`: LLM/VLM API model identifier  
- `TEMPERATURE`: generation temperature for the LLM

**Output**
- A configuration dictionary printed for reproducibility and logging.


### 2. Load COCO & Sample Images

In this step, we load the COCO dataset using the official API, sample a subset of images with a fixed random seed, and perform a visual sanity check by overlaying bounding boxes on a few example images.

**Steps**
- Initialize the COCO API with instance annotations
- Randomly sample a subset of images
- Visualize 3–5 sampled images with bounding box overlays

**Output**
- Sample visualizations to verify correct loading and annotation parsing

In [2]:
from pycocotools.coco import COCO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np
import random
import os

# ----------------------------
# Initialize COCO API
# ----------------------------
coco = COCO(annFile)

img_ids = coco.getImgIds()
random.seed(42)
sampled_img_ids = random.sample(img_ids, min(N_IMAGES, len(img_ids)))

print(f"Total COCO images: {len(img_ids)}")
print(f"Sampled images: {len(sampled_img_ids)}")

# ----------------------------
# Visual Sanity Check
# ----------------------------
def visualize_coco_sample(img_id):
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(COCO_ROOT, img_info["file_name"])
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    for ann in anns:
        x, y, w, h = ann["bbox"]
        if w * h < MIN_BBOX_AREA:
            continue
        draw.rectangle([x, y, x + w, y + h], outline="red", width=2)

    return image

# Show 3–5 sample images
num_vis = min(5, len(sampled_img_ids))
plt.figure(figsize=(15, 5))

for i, img_id in enumerate(sampled_img_ids[:num_vis]):
    plt.subplot(1, num_vis, i + 1)
    plt.imshow(visualize_coco_sample(img_id))
    plt.axis("off")
    plt.title(f"Image ID: {img_id}")

plt.tight_layout()
plt.show()

NameError: name 'annFile' is not defined